In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GEMINI_API_KEY"]=os.getenv("GEMINI_API_KEY")
#Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACKING_V2"]="true"
os.environ["LANGCHAIN_PROJECT2"]=os.getenv("LANGCHAIN_PROJECT2")

In [2]:
import sys
print(sys.executable)

c:\Users\shiva\anaconda3\envs\genai_env\python.exe


In [3]:
%pip install -U langchain-classic

Note: you may need to restart the kernel to use updated packages.


In [4]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

print("Import successful!")

Import successful!


In [5]:
##Data Ingestion--From the website we need to scrap the data
from langchain_community.document_loaders import WebBaseLoader

C:\Users\shiva\AppData\Local\Temp\ipykernel_13160\1311322469.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
loader=WebBaseLoader("https://www.langchain.com/langchain")
loader

In [7]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://www.langchain.com/langchain', 'title': 'LangChain: Open Source AI Agent Framework for Any Model', 'description': 'Build agents in minutes with LangChain, the open source framework with pre-built agent architectures and 1,000+ integrations.', 'language': 'en'}, page_content="LangChain: Open Source AI Agent Framework for Any Model\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nWould you notice if your agent got worse in production? Tell us in our annual survey for the chance to win LangChain swag.|Take survey ↗\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nProducts\n\nLangSmith Platform\n\nAgent ImprovementEngineImprove agents autonomouslyObservabilitySee exactly what your agents are doingEvaluationScore and improve agent performanceAgent InfrastructureDeploymentShip and scale agents in productionSandboxesRun agent-generated code safelyLLM GatewayControl agent model callsNo-Code AgentsFleetAgents for the whole companyOpen Source\n\nAgent Framew

In [8]:
##Load data-->docs-->Divide our texts to chunks-->vectors-->vector embeddings-->Vector Store
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=20)
documents=text_splitter.split_documents(docs)


In [9]:
docs

[Document(metadata={'source': 'https://www.langchain.com/langchain', 'title': 'LangChain: Open Source AI Agent Framework for Any Model', 'description': 'Build agents in minutes with LangChain, the open source framework with pre-built agent architectures and 1,000+ integrations.', 'language': 'en'}, page_content="LangChain: Open Source AI Agent Framework for Any Model\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nWould you notice if your agent got worse in production? Tell us in our annual survey for the chance to win LangChain swag.|Take survey ↗\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nProducts\n\nLangSmith Platform\n\nAgent ImprovementEngineImprove agents autonomouslyObservabilitySee exactly what your agents are doingEvaluationScore and improve agent performanceAgent InfrastructureDeploymentShip and scale agents in productionSandboxesRun agent-generated code safelyLLM GatewayControl agent model callsNo-Code AgentsFleetAgents for the whole companyOpen Source\n\nAgent Framew

In [10]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [11]:
from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)

In [12]:
vectorstoredb

In [13]:
#Query from the vectordb
query="Langsmith has two usage limits:total traces and extended"
result=vectorstoredb.similarity_search(query)
result[0].page_content

'do I use LangChain with LangSmith?'

In [14]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash"
)

In [17]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
    Answer the following question based on the context:

    <context>
    {context}
    </context>

    Question: {input}
    """
)

document_chain = create_stuff_documents_chain(llm, prompt)

In [18]:
from langchain_core.documents import Document

document_chain.invoke({
    "input": "What are the two usage limits of LangSmith?",
    "context": [
        Document(
            page_content="LangSmith has two usage limits: total traces and extended."
        )
    ]
})

'Based on the context, the two usage limits of LangSmith are **total traces** and **extended**.'

However we want the documents to first come from the retriever we just setup.That way,we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [19]:
#Input--->Retriever-->vectorstoredb
vectorstoredb

In [21]:
retriever=vectorstoredb.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)

In [22]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020BF98CFA50>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n    Answer the following question based on the context:\n\n    <context>\n    {context}\n    </context>\n\n    Questi

In [23]:
#Get the response from the llm
response=retrieval_chain.invoke({"input": "LangSmith has two usage limits: total traces and extended."})

In [24]:
response

{'input': 'LangSmith has two usage limits: total traces and extended.',
 'context': [Document(id='c1189ad2-1b2c-48b0-8cf6-5dcb9401cf57', metadata={'source': 'https://www.langchain.com/langchain', 'title': 'LangChain: Open Source AI Agent Framework for Any Model', 'description': 'Build agents in minutes with LangChain, the open source framework with pre-built agent architectures and 1,000+ integrations.', 'language': 'en'}, page_content='variable to get started.See what your agent is really doingLangSmith, our agent engineering'),
  Document(id='787deb1c-775a-4fca-b8c6-6a7e2334f065', metadata={'source': 'https://www.langchain.com/langchain', 'title': 'LangChain: Open Source AI Agent Framework for Any Model', 'description': 'Build agents in minutes with LangChain, the open source framework with pre-built agent architectures and 1,000+ integrations.', 'language': 'en'}, page_content='Get a demo\n\nTry LangSmith\n\nGet a demo'),
  Document(id='bef56046-1106-4c04-940f-d4d1040aabda', metadat

In [25]:
response['answer']

"Based on the provided context, there is no information about LangSmith's usage limits, total traces, or extended limits."

In [26]:
response['context']

[Document(id='c1189ad2-1b2c-48b0-8cf6-5dcb9401cf57', metadata={'source': 'https://www.langchain.com/langchain', 'title': 'LangChain: Open Source AI Agent Framework for Any Model', 'description': 'Build agents in minutes with LangChain, the open source framework with pre-built agent architectures and 1,000+ integrations.', 'language': 'en'}, page_content='variable to get started.See what your agent is really doingLangSmith, our agent engineering'),
 Document(id='787deb1c-775a-4fca-b8c6-6a7e2334f065', metadata={'source': 'https://www.langchain.com/langchain', 'title': 'LangChain: Open Source AI Agent Framework for Any Model', 'description': 'Build agents in minutes with LangChain, the open source framework with pre-built agent architectures and 1,000+ integrations.', 'language': 'en'}, page_content='Get a demo\n\nTry LangSmith\n\nGet a demo'),
 Document(id='bef56046-1106-4c04-940f-d4d1040aabda', metadata={'source': 'https://www.langchain.com/langchain', 'title': 'LangChain: Open Source A